# 🗄️ Vector Databases: Where Meaning Lives

## 🎯 What You'll Learn
- What vector databases are and why regular databases aren't enough
- How vector search works under the hood
- Key indexing algorithms (Flat, IVF, HNSW)
- Comparison of popular vector databases
- Build a mini vector database from scratch!

**Prerequisites:** Understanding of embeddings ([Notebook 01](01_what_is_rag.ipynb))

---

## 🤔 Why Regular Databases Aren't Enough

Imagine you have a **regular dictionary** at home. If you want to find the word "cat", you flip to the C section, then CA, then CAT — **exact lookup**. That's how a regular (SQL) database works. You ask for *exactly* what you want, and it gives you *exactly* that match.

Now imagine instead of a dictionary, you have a **brilliant librarian** 📚. You walk in and say, *"I want something about fluffy animals that purr."* The librarian doesn't need you to say "cat" — they **understand the meaning** behind your words and hand you the perfect book. That's a **vector database**.

### SQL Database vs. Vector Database

| Feature | SQL Database 🗃️ | Vector Database 🧠 |
|---|---|---|
| **Search Type** | Exact keyword match | Semantic similarity |
| **Query** | `WHERE title = 'solar panels'` | "find renewable energy" |
| **Matches** | Only rows with exact text | Solar panels, wind turbines, green energy... |
| **Data Stored** | Rows & columns | High-dimensional vectors |
| **Lookup Speed** | O(log n) with B-tree index | O(log n) with ANN index |
| **Understands Meaning?** | ❌ No | ✅ Yes |

> **Key Insight:** When you search for "renewable energy" in a SQL database, it will **never** find a document titled "solar panel efficiency" — because the words don't match. A vector database finds it instantly because the **meanings** are close together in vector space.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left: Regular DB (Keyword Match) ---
documents = ['solar panels', 'wind energy', 'coal mining', 'renewable power', 'oil drilling', 'green tech']
doc_y = list(range(len(documents)))
colors_left = ['#cccccc', '#cccccc', '#cccccc', '#cccccc', '#cccccc', '#cccccc']

# Only exact match for 'solar panels'
match_idx = 0  # 'solar panels' matches keyword 'solar'
colors_left[match_idx] = '#2ecc71'

ax1.barh(doc_y, [1]*len(documents), color=colors_left, edgecolor='white', height=0.6)
for i, doc in enumerate(documents):
    ax1.text(0.5, i, doc, ha='center', va='center', fontsize=12, fontweight='bold')

ax1.set_title('Regular DB: Search "solar"', fontsize=14, fontweight='bold')
ax1.set_xlabel('Only exact keyword matches', fontsize=11)
ax1.set_yticks([])
ax1.set_xticks([])
ax1.annotate('✅ Match!', xy=(0.85, 0), fontsize=12, color='green', fontweight='bold')
for i in range(1, len(documents)):
    ax1.annotate('❌ No match', xy=(0.82, i), fontsize=10, color='red')

# --- Right: Vector DB (Semantic Similarity) ---
np.random.seed(42)
# Cluster: renewable energy topics (close together)
renewable_x = [2.0, 2.3, 2.5, 2.1]
renewable_y = [3.0, 3.4, 2.7, 3.6]
renewable_labels = ['solar panels', 'wind energy', 'renewable power', 'green tech']

# Cluster: fossil fuels (far away)
fossil_x = [7.0, 7.5]
fossil_y = [7.0, 6.5]
fossil_labels = ['coal mining', 'oil drilling']

# Query point
query_x, query_y = 2.2, 3.2

ax2.scatter(renewable_x, renewable_y, c='#2ecc71', s=200, zorder=5, edgecolors='black', linewidth=1.5)
ax2.scatter(fossil_x, fossil_y, c='#e74c3c', s=200, zorder=5, edgecolors='black', linewidth=1.5)
ax2.scatter([query_x], [query_y], c='#f39c12', s=300, marker='*', zorder=10, edgecolors='black', linewidth=1.5)

for x, y, label in zip(renewable_x, renewable_y, renewable_labels):
    ax2.annotate(label, (x, y), textcoords='offset points', xytext=(10, 10), fontsize=10)
for x, y, label in zip(fossil_x, fossil_y, fossil_labels):
    ax2.annotate(label, (x, y), textcoords='offset points', xytext=(10, 10), fontsize=10)
ax2.annotate('Query: "renewable\nenergy"', (query_x, query_y), textcoords='offset points',
             xytext=(-80, -40), fontsize=11, fontweight='bold', color='#e67e22',
             arrowprops=dict(arrowstyle='->', color='#e67e22', lw=2))

# Draw circles showing nearest neighbors
circle = plt.Circle((query_x, query_y), 0.8, fill=False, linestyle='--', color='#f39c12', linewidth=2)
ax2.add_patch(circle)

ax2.set_title('Vector DB: Search "renewable energy"', fontsize=14, fontweight='bold')
ax2.set_xlabel('Finds semantically similar documents!', fontsize=11)
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.set_xticks([])
ax2.set_yticks([])

green_patch = mpatches.Patch(color='#2ecc71', label='Relevant (nearby)')
red_patch = mpatches.Patch(color='#e74c3c', label='Not relevant (far away)')
ax2.legend(handles=[green_patch, red_patch], loc='upper left', fontsize=10)

plt.tight_layout()
plt.savefig('figures/03_regular_vs_vector_db.png', dpi=150, bbox_inches='tight')
plt.show()
print("Left: Regular DB only finds exact keyword matches.")
print("Right: Vector DB finds ALL semantically related documents!")

---

## ⚙️ How Vector Databases Work

A vector database does four main things:

### 📥 Step 1: Store Vectors
When you add a document, it gets converted into a **vector** (a list of numbers representing its meaning) and stored alongside any **metadata** (author, date, category, etc.).

### 📊 Step 2: Build an Index
The database organizes vectors into a **searchable structure** — like sorting books in a library by topic. This is the secret sauce that makes search fast.

### 🔍 Step 3: Search by Similarity
When you query, your question is also converted to a vector. The database finds the **nearest vectors** to your query — these are the most semantically similar documents.

### 📤 Step 4: Return Results
The database returns the **top-K** closest documents along with their similarity scores and metadata.

```
Query: "How do rockets work?"
   ↓
[0.23, 0.87, -0.12, ...]  ← Query vector
   ↓
🔍 Search index for nearest neighbors
   ↓
Results:
  1. "Rocket propulsion principles" (score: 0.94)
  2. "Space shuttle engine design" (score: 0.89)
  3. "Thrust and Newton's third law" (score: 0.85)
```

---

## 🏠 The Nearest Neighbor Problem

At its core, vector search is about finding the **nearest neighbors** — the vectors closest to your query.

### The Brute Force Approach

The simplest method: compare your query against **every single vector** in the database and keep the closest ones.

**Analogy for a 12-year-old:** Imagine you're trying to find your friend in a city of 1 million houses. The brute force approach is knocking on **every single door** and asking "Is my friend here?" 🚪🚪🚪

Sure, you'll definitely find them — but it'll take **forever**.

| Number of Vectors | Dimensions | Comparisons Needed | Time (approx) |
|---|---|---|---|
| 1,000 | 768 | 1,000 | ~0.1ms |
| 1,000,000 | 768 | 1,000,000 | ~100ms |
| 1,000,000,000 | 768 | 1,000,000,000 | ~100 seconds 🐌 |

> **Problem:** Brute force has **O(n × d)** complexity. With billions of vectors, this is way too slow. We need **smarter** ways to search!

In [ ]:
import numpy as np

class MiniVectorDB:
    """A minimal vector database built from scratch with numpy."""
    
    def __init__(self):
        self.documents = []    # Store document text
        self.vectors = []      # Store embedding vectors
    
    def add(self, doc: str, vector: np.ndarray):
        """Add a document and its vector to the database."""
        self.documents.append(doc)
        self.vectors.append(vector)
    
    def _cosine_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        """Compute cosine similarity between two vectors."""
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    
    def search(self, query_vector: np.ndarray, top_k: int = 3):
        """Search for the top-K most similar documents."""
        similarities = []
        for i, vec in enumerate(self.vectors):
            sim = self._cosine_similarity(query_vector, vec)
            similarities.append((sim, i))
        
        # Sort by similarity (highest first)
        similarities.sort(reverse=True)
        
        results = []
        for sim, idx in similarities[:top_k]:
            results.append({
                'document': self.documents[idx],
                'similarity': round(sim, 4),
                'index': idx
            })
        return results

# --- Create our mini database with clustered embeddings ---
np.random.seed(42)
db = MiniVectorDB()

# Animal documents — vectors clustered around [1, 0, 0, 0, 0]
animal_docs = [
    "Dogs are loyal companions and love playing fetch",
    "Cats are independent creatures that purr when happy",
    "Dolphins are intelligent marine mammals",
]
for doc in animal_docs:
    vec = np.array([1.0, 0.0, 0.0, 0.0, 0.0]) + np.random.normal(0, 0.15, 5)
    db.add(doc, vec)

# Space documents — vectors clustered around [0, 1, 0, 0, 0]
space_docs = [
    "The Mars rover explores the red planet surface",
    "Black holes warp spacetime and trap light",
    "Astronauts train for years before going to space",
    "The James Webb telescope captures distant galaxies",
]
for doc in space_docs:
    vec = np.array([0.0, 1.0, 0.0, 0.0, 0.0]) + np.random.normal(0, 0.15, 5)
    db.add(doc, vec)

# Cooking documents — vectors clustered around [0, 0, 1, 0, 0]
cooking_docs = [
    "The best pasta recipes use fresh tomatoes and basil",
    "Baking bread requires patience and good yeast",
    "Sushi chefs train for a decade to master their craft",
]
for doc in cooking_docs:
    vec = np.array([0.0, 0.0, 1.0, 0.0, 0.0]) + np.random.normal(0, 0.15, 5)
    db.add(doc, vec)

print(f"📦 Database loaded with {len(db.documents)} documents\n")

# --- Search! ---
# Query about space (should match space docs)
query = np.array([0.05, 0.95, 0.05, 0.0, 0.0])  # Close to space cluster
print("🔍 Query: 'Tell me about space exploration'")
print("=" * 50)
results = db.search(query, top_k=3)
for i, r in enumerate(results):
    print(f"  #{i+1} (similarity: {r['similarity']:.4f}): {r['document']}")

print()

# Query about animals (should match animal docs)
query2 = np.array([0.9, 0.05, 0.1, 0.0, 0.0])  # Close to animal cluster
print("🔍 Query: 'What are some cute pets?'")
print("=" * 50)
results2 = db.search(query2, top_k=3)
for i, r in enumerate(results2):
    print(f"  #{i+1} (similarity: {r['similarity']:.4f}): {r['document']}")

print()

# Query about cooking (should match cooking docs)
query3 = np.array([0.0, 0.1, 0.9, 0.05, 0.0])  # Close to cooking cluster
print("🔍 Query: 'How to cook Italian food?'")
print("=" * 50)
results3 = db.search(query3, top_k=3)
for i, r in enumerate(results3):
    print(f"  #{i+1} (similarity: {r['similarity']:.4f}): {r['document']}")

---

## 🏗️ Indexing Algorithms: Making Search Fast

Instead of checking every vector (brute force), smart indexing algorithms let us skip most of the database and only check a small fraction. Here are the three most important ones:

### 1️⃣ Flat Index (Brute Force)
- **How it works:** Compare the query against *every* vector in the database.
- **Analogy:** Checking **every house** in the city. 🏘️🏘️🏘️
- **Accuracy:** 100% — you *will* find the true nearest neighbor.
- **Speed:** Slow — O(n) comparisons.
- **Best for:** Small datasets (< 50K vectors).

### 2️⃣ IVF (Inverted File Index)
- **How it works:** Divide vectors into **clusters** (using k-means). At search time, only check vectors in the **nearest cluster(s)**.
- **Analogy:** The city is divided into **neighborhoods** 🏘️. First find the right neighborhood, then check houses only there.
- **Accuracy:** ~95–99% (might miss some edge cases).
- **Speed:** Fast — only check 1–10% of vectors.
- **Best for:** Medium datasets (50K–10M vectors).

### 3️⃣ HNSW (Hierarchical Navigable Small World)
- **How it works:** Build a **multi-layer graph** where each node is connected to similar nodes. Search by "hopping" through friends-of-friends.
- **Analogy:** Finding someone at a party 🎉. You ask a friend → they point you to someone closer → that person points you even closer → you find them!
- **Accuracy:** ~95–99.9% with proper tuning.
- **Speed:** Very fast — O(log n) hops.
- **Best for:** Large datasets where speed matters.

| Algorithm | Speed | Accuracy | Memory | Build Time |
|---|---|---|---|---|
| Flat | 🐌 Slow | 💯 Perfect | Low | None |
| IVF | 🏃 Fast | 🎯 ~95-99% | Medium | Medium |
| HNSW | 🚀 Very Fast | 🎯 ~95-99.9% | High | Slow |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

# Generate clustered 2D data
n_per_cluster = 20
centers = [(2, 2), (7, 3), (4, 7), (8, 8)]
cluster_colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
all_points = []
all_cluster_ids = []
for i, (cx, cy) in enumerate(centers):
    pts = np.random.normal(0, 0.6, (n_per_cluster, 2)) + np.array([cx, cy])
    all_points.append(pts)
    all_cluster_ids.extend([i] * n_per_cluster)
all_points = np.vstack(all_points)
all_cluster_ids = np.array(all_cluster_ids)
query = np.array([5.0, 6.5])

# --- Flat Index ---
# Highlight ALL points as searched
ax1.scatter(all_points[:, 0], all_points[:, 1], c='#f39c12', s=60, alpha=0.7,
            edgecolors='orange', linewidth=0.5, label='Searched')
ax1.scatter([query[0]], [query[1]], c='red', s=200, marker='*', zorder=10,
            edgecolors='black', linewidth=1.5, label='Query')

# Find true nearest neighbors
dists = np.linalg.norm(all_points - query, axis=1)
nn_idx = np.argsort(dists)[:3]
ax1.scatter(all_points[nn_idx, 0], all_points[nn_idx, 1], c='#2ecc71', s=120,
            edgecolors='black', linewidth=2, zorder=8)
for idx in nn_idx:
    ax1.plot([query[0], all_points[idx, 0]], [query[1], all_points[idx, 1]],
             'g--', alpha=0.6, linewidth=1.5)

ax1.set_title('Flat (Brute Force)', fontsize=13, fontweight='bold')
ax1.text(0.5, -0.08, f'Searches ALL {len(all_points)} vectors', transform=ax1.transAxes,
         ha='center', fontsize=10, color='#e67e22')
ax1.set_xlim(-0.5, 10.5)
ax1.set_ylim(-0.5, 10.5)
ax1.set_xticks([])
ax1.set_yticks([])

# --- IVF Index ---
for i, (cx, cy) in enumerate(centers):
    cluster_mask = all_cluster_ids == i
    ax2.scatter(all_points[cluster_mask, 0], all_points[cluster_mask, 1],
                c='#cccccc', s=60, alpha=0.5, edgecolors='grey', linewidth=0.5)
    # Draw cluster boundary
    circle = plt.Circle((cx, cy), 1.8, fill=False, linestyle='--',
                         color=cluster_colors[i], linewidth=1.5, alpha=0.6)
    ax2.add_patch(circle)
    ax2.scatter([cx], [cy], c=cluster_colors[i], s=100, marker='x', linewidths=3, zorder=6)

# Highlight only nearest cluster (cluster 2: center at (4,7))
nearest_cluster = 2
cluster_mask = all_cluster_ids == nearest_cluster
ax2.scatter(all_points[cluster_mask, 0], all_points[cluster_mask, 1],
            c='#f39c12', s=60, alpha=0.9, edgecolors='orange', linewidth=1, label='Searched')

# Draw emphasis on the nearest cluster
circle_highlight = plt.Circle(centers[nearest_cluster], 1.8, fill=True, alpha=0.1,
                               color='#f39c12', linewidth=2)
ax2.add_patch(circle_highlight)

ax2.scatter([query[0]], [query[1]], c='red', s=200, marker='*', zorder=10,
            edgecolors='black', linewidth=1.5)
# Show nearest neighbors in that cluster
cluster_points = all_points[cluster_mask]
cluster_dists = np.linalg.norm(cluster_points - query, axis=1)
local_nn = np.argsort(cluster_dists)[:3]
ax2.scatter(cluster_points[local_nn, 0], cluster_points[local_nn, 1], c='#2ecc71', s=120,
            edgecolors='black', linewidth=2, zorder=8)

ax2.set_title('IVF (Cluster-Based)', fontsize=13, fontweight='bold')
ax2.text(0.5, -0.08, f'Searches only {n_per_cluster} of {len(all_points)} vectors!',
         transform=ax2.transAxes, ha='center', fontsize=10, color='#e67e22')
ax2.set_xlim(-0.5, 10.5)
ax2.set_ylim(-0.5, 10.5)
ax2.set_xticks([])
ax2.set_yticks([])

# --- HNSW Index ---
# Show multi-layer graph traversal
ax3.scatter(all_points[:, 0], all_points[:, 1], c='#cccccc', s=40, alpha=0.4,
            edgecolors='grey', linewidth=0.5)
ax3.scatter([query[0]], [query[1]], c='red', s=200, marker='*', zorder=10,
            edgecolors='black', linewidth=1.5)

# Simulate graph traversal path
path_points = [
    (1.5, 8.5),  # Entry point (top layer)
    (3.5, 7.0),  # Hop 1
    (4.2, 6.8),  # Hop 2
    (4.5, 7.1),  # Hop 3 — near query
]

# Draw the path
for i in range(len(path_points)):
    px, py = path_points[i]
    color = '#3498db' if i < len(path_points) - 1 else '#2ecc71'
    ax3.scatter([px], [py], c=color, s=100, edgecolors='black', linewidth=1.5, zorder=7)
    if i > 0:
        prev_x, prev_y = path_points[i-1]
        ax3.annotate('', xy=(px, py), xytext=(prev_x, prev_y),
                     arrowprops=dict(arrowstyle='->', color='#3498db', lw=2.5))

# Arrow from last path point to query
ax3.annotate('', xy=(query[0], query[1]), xytext=(path_points[-1][0], path_points[-1][1]),
             arrowprops=dict(arrowstyle='->', color='#2ecc71', lw=2.5))

# Draw some graph edges (background)
edge_pairs = [(0, 5), (5, 12), (12, 18), (3, 8), (8, 15), (15, 22),
              (1, 6), (6, 14), (10, 19), (19, 25), (7, 16)]
for i, j in edge_pairs:
    if i < len(all_points) and j < len(all_points):
        ax3.plot([all_points[i, 0], all_points[j, 0]],
                 [all_points[i, 1], all_points[j, 1]],
                 'grey', alpha=0.15, linewidth=0.8)

ax3.annotate('Entry\npoint', path_points[0], textcoords='offset points',
             xytext=(-40, 10), fontsize=9, color='#3498db', fontweight='bold')
ax3.annotate('Hop through\ngraph edges', (3.0, 7.5), fontsize=9, color='#3498db',
             fontstyle='italic')

ax3.set_title('HNSW (Graph-Based)', fontsize=13, fontweight='bold')
ax3.text(0.5, -0.08, f'Only {len(path_points)} hops instead of {len(all_points)} comparisons!',
         transform=ax3.transAxes, ha='center', fontsize=10, color='#e67e22')
ax3.set_xlim(-0.5, 10.5)
ax3.set_ylim(-0.5, 10.5)
ax3.set_xticks([])
ax3.set_yticks([])

plt.tight_layout()
plt.savefig('figures/03_indexing_algorithms.png', dpi=150, bbox_inches='tight')
plt.show()
print("Flat: checks everything. IVF: checks one cluster. HNSW: hops through a graph.")

In [ ]:
import numpy as np

class IVFIndex:
    """Simplified Inverted File Index using k-means clustering."""
    
    def __init__(self, n_clusters: int = 4):
        self.n_clusters = n_clusters
        self.centroids = None
        self.clusters = {}  # cluster_id -> list of (index, vector)
        self.documents = []
        self.vectors = []
    
    def add(self, doc: str, vector: np.ndarray):
        self.documents.append(doc)
        self.vectors.append(vector)
    
    def build(self, n_iterations: int = 20):
        """Build the IVF index using simple k-means."""
        vectors = np.array(self.vectors)
        n = len(vectors)
        
        # Initialize centroids randomly
        indices = np.random.choice(n, self.n_clusters, replace=False)
        self.centroids = vectors[indices].copy()
        
        # K-means iterations
        for _ in range(n_iterations):
            # Assign each vector to nearest centroid
            assignments = []
            for v in vectors:
                dists = [np.linalg.norm(v - c) for c in self.centroids]
                assignments.append(np.argmin(dists))
            
            # Update centroids
            for c in range(self.n_clusters):
                members = vectors[[i for i, a in enumerate(assignments) if a == c]]
                if len(members) > 0:
                    self.centroids[c] = members.mean(axis=0)
        
        # Build inverted lists
        self.clusters = {c: [] for c in range(self.n_clusters)}
        for i, v in enumerate(vectors):
            dists = [np.linalg.norm(v - c) for c in self.centroids]
            cluster_id = np.argmin(dists)
            self.clusters[cluster_id].append((i, v))
        
        print(f"Built IVF index with {self.n_clusters} clusters:")
        for c, members in self.clusters.items():
            print(f"  Cluster {c}: {len(members)} vectors")
    
    def search(self, query_vector: np.ndarray, top_k: int = 3, n_probe: int = 1):
        """Search by only checking the nearest cluster(s)."""
        # Find nearest centroid(s)
        centroid_dists = [np.linalg.norm(query_vector - c) for c in self.centroids]
        nearest_clusters = np.argsort(centroid_dists)[:n_probe]
        
        # Search only within those clusters
        candidates = []
        vectors_searched = 0
        for cluster_id in nearest_clusters:
            for idx, vec in self.clusters[cluster_id]:
                sim = np.dot(query_vector, vec) / (np.linalg.norm(query_vector) * np.linalg.norm(vec))
                candidates.append((sim, idx))
                vectors_searched += 1
        
        candidates.sort(reverse=True)
        
        results = []
        for sim, idx in candidates[:top_k]:
            results.append({
                'document': self.documents[idx],
                'similarity': round(sim, 4)
            })
        
        return results, vectors_searched


# --- Demo ---
np.random.seed(42)
ivf = IVFIndex(n_clusters=4)

# Add 80 documents across 4 topics
topics = {
    'animals': np.array([1, 0, 0, 0, 0], dtype=float),
    'space': np.array([0, 1, 0, 0, 0], dtype=float),
    'cooking': np.array([0, 0, 1, 0, 0], dtype=float),
    'sports': np.array([0, 0, 0, 1, 0], dtype=float),
}

for topic, base_vec in topics.items():
    for i in range(20):
        vec = base_vec + np.random.normal(0, 0.15, 5)
        ivf.add(f"{topic}_doc_{i}", vec)

print("\n📊 Building IVF index...")
ivf.build()

# Search
query = np.array([0.05, 0.95, 0.0, 0.05, 0.0])  # Space-like query
print("\n🔍 Searching for 'space exploration'...")

# IVF search (1 probe)
results_ivf, searched_ivf = ivf.search(query, top_k=3, n_probe=1)
print(f"\n--- IVF (1 probe) ---")
print(f"Vectors searched: {searched_ivf} / {len(ivf.vectors)} ({100*searched_ivf/len(ivf.vectors):.0f}%)")
for r in results_ivf:
    print(f"  {r['document']} (sim: {r['similarity']})")

# IVF search (2 probes)
results_ivf2, searched_ivf2 = ivf.search(query, top_k=3, n_probe=2)
print(f"\n--- IVF (2 probes) ---")
print(f"Vectors searched: {searched_ivf2} / {len(ivf.vectors)} ({100*searched_ivf2/len(ivf.vectors):.0f}%)")
for r in results_ivf2:
    print(f"  {r['document']} (sim: {r['similarity']})")

# Flat comparison
print(f"\n--- Flat (brute force) ---")
print(f"Vectors searched: {len(ivf.vectors)} / {len(ivf.vectors)} (100%)")
print(f"\n💡 IVF searched only {100*searched_ivf/len(ivf.vectors):.0f}% of vectors but found the right cluster!")

---

## 🌐 Popular Vector Databases Compared

There's an explosion of vector databases. Here's how the most popular ones compare:

| Database | Type | Best For | Indexing | Language | Open Source? |
|---|---|---|---|---|---|
| **FAISS** | Library | Research, prototyping | Flat, IVF, HNSW, PQ | C++/Python | ✅ Yes (Meta) |
| **ChromaDB** | Embedded DB | Quick prototyping, small-medium scale | HNSW | Python | ✅ Yes |
| **Pinecone** | Managed service | Production, zero-ops | Proprietary | API-based | ❌ No |
| **Weaviate** | Self-hosted/Cloud | ML-native apps, multi-modal | HNSW | Go | ✅ Yes |
| **Qdrant** | Self-hosted/Cloud | High-performance filtering | HNSW | Rust | ✅ Yes |
| **Milvus** | Distributed | Massive scale (billions) | IVF, HNSW, DiskANN | Go/C++ | ✅ Yes (Zilliz) |

### 🤔 How to Choose?

```
Just learning / prototyping?  →  ChromaDB or FAISS
Need managed service?         →  Pinecone
Need open-source + filtering? →  Qdrant or Weaviate  
Billions of vectors?          →  Milvus
Research / benchmarking?      →  FAISS
```

---

## 🔑 Key Concepts in Vector Databases

### 📏 Dimensions
The **size** of each vector. Common embedding models produce vectors of different sizes:
- **OpenAI text-embedding-3-small:** 1,536 dimensions
- **Cohere embed-v3:** 1,024 dimensions
- **all-MiniLM-L6-v2:** 384 dimensions
- **BGE-large:** 1,024 dimensions

> More dimensions = more expressive, but more memory and slower search.

### 📐 Distance Metrics
How we measure "closeness" between vectors:

| Metric | Formula | Range | Best For |
|---|---|---|---|
| **Cosine Similarity** | cos(θ) = (A·B)/(‖A‖‖B‖) | [-1, 1] | Text similarity (most common) |
| **Euclidean (L2)** | √Σ(aᵢ - bᵢ)² | [0, ∞) | When magnitude matters |
| **Dot Product** | A·B = Σ(aᵢ × bᵢ) | (-∞, ∞) | When vectors are normalized |

### 🏷️ Metadata Filtering
Vector databases let you attach **metadata** (tags, categories, dates) to vectors and filter results:
```python
# "Find documents similar to my query, but only from 2024"
results = db.search(
    query_vector=query,
    filter={"year": 2024, "category": "science"}
)
```

### 🏆 Top-K
The number of nearest results to return. Typical values:
- **RAG applications:** top_k = 3–5
- **Recommendation systems:** top_k = 10–50
- **Deduplication:** top_k = 1

In [ ]:
import numpy as np

class MiniVectorDBWithMetadata:
    """Vector database with metadata filtering support."""
    
    def __init__(self):
        self.documents = []
        self.vectors = []
        self.metadata = []
    
    def add(self, doc: str, vector: np.ndarray, metadata: dict = None):
        self.documents.append(doc)
        self.vectors.append(vector)
        self.metadata.append(metadata or {})
    
    def search(self, query_vector: np.ndarray, top_k: int = 3, filter_dict: dict = None):
        """Search with optional metadata filtering."""
        candidates = []
        
        for i, (vec, meta) in enumerate(zip(self.vectors, self.metadata)):
            # Apply metadata filter
            if filter_dict:
                match = all(meta.get(k) == v for k, v in filter_dict.items())
                if not match:
                    continue
            
            # Compute similarity
            sim = np.dot(query_vector, vec) / (np.linalg.norm(query_vector) * np.linalg.norm(vec))
            candidates.append((sim, i))
        
        candidates.sort(reverse=True)
        
        results = []
        for sim, idx in candidates[:top_k]:
            results.append({
                'document': self.documents[idx],
                'similarity': round(sim, 4),
                'metadata': self.metadata[idx]
            })
        return results


# --- Build a database with metadata ---
np.random.seed(42)
db = MiniVectorDBWithMetadata()

# Science articles
science_base = np.array([1.0, 0.0, 0.0, 0.0])
db.add("Quantum computing breakthrough in 2024",
       science_base + np.random.normal(0, 0.1, 4),
       {"category": "science", "year": 2024})
db.add("CRISPR gene editing advances",
       science_base + np.random.normal(0, 0.1, 4),
       {"category": "science", "year": 2023})
db.add("New particle discovered at CERN",
       science_base + np.random.normal(0, 0.1, 4),
       {"category": "science", "year": 2024})
db.add("Fusion reactor milestone achieved",
       science_base + np.random.normal(0, 0.1, 4),
       {"category": "science", "year": 2023})

# Technology articles
tech_base = np.array([0.0, 1.0, 0.0, 0.0])
db.add("GPT-5 released with multimodal capabilities",
       tech_base + np.random.normal(0, 0.1, 4),
       {"category": "technology", "year": 2024})
db.add("Apple Vision Pro sales report",
       tech_base + np.random.normal(0, 0.1, 4),
       {"category": "technology", "year": 2024})
db.add("Self-driving cars reach level 4 autonomy",
       tech_base + np.random.normal(0, 0.1, 4),
       {"category": "technology", "year": 2023})

# Sports articles
sports_base = np.array([0.0, 0.0, 1.0, 0.0])
db.add("Olympics 2024 breaks viewership records",
       sports_base + np.random.normal(0, 0.1, 4),
       {"category": "sports", "year": 2024})
db.add("World Cup 2023 final highlights",
       sports_base + np.random.normal(0, 0.1, 4),
       {"category": "sports", "year": 2023})

print(f"📦 Database has {len(db.documents)} documents\n")

# --- Search without filter ---
query = science_base + np.random.normal(0, 0.05, 4)
print("🔍 Search: 'latest science discoveries' (NO filter)")
print("=" * 55)
results = db.search(query, top_k=3)
for r in results:
    print(f"  [{r['metadata']['category']}, {r['metadata']['year']}] "
          f"{r['document']} (sim: {r['similarity']})")

print()

# --- Search WITH filter: only 2024 ---
print("🔍 Search: 'latest science discoveries' (filter: year=2024)")
print("=" * 55)
results_filtered = db.search(query, top_k=3, filter_dict={"year": 2024})
for r in results_filtered:
    print(f"  [{r['metadata']['category']}, {r['metadata']['year']}] "
          f"{r['document']} (sim: {r['similarity']})")

print()

# --- Search WITH filter: only technology ---
print("🔍 Search: 'latest science discoveries' (filter: category=technology)")
print("=" * 55)
results_tech = db.search(query, top_k=3, filter_dict={"category": "technology"})
for r in results_tech:
    print(f"  [{r['metadata']['category']}, {r['metadata']['year']}] "
          f"{r['document']} (sim: {r['similarity']})")

print("\n💡 Metadata filtering lets you combine semantic search with precise constraints!")

---

## 📊 Performance: The Recall vs. Speed Tradeoff

In the world of vector search, there's always a tradeoff:

- **Recall** = What fraction of the *true* nearest neighbors did we actually find? (Higher is better)
- **Speed** = How many queries per second can we handle? (Higher is better)

**Analogy:** Imagine searching for your keys 🔑
- **High recall, low speed:** Check every pocket, every drawer, under every cushion. You *will* find them, but it takes 20 minutes.
- **Low recall, high speed:** Only check your jacket pocket. Takes 2 seconds, but you might miss them if they're in your backpack.
- **The sweet spot:** Check the 3 most likely places first. Fast AND usually finds them.

### 💾 Memory Calculation

How much memory does a vector database need?

```
Memory = num_vectors × dimensions × bytes_per_float

Example: 1 million vectors, 1536 dimensions, float32 (4 bytes)
= 1,000,000 × 1,536 × 4 bytes
= 6.144 GB (just for vectors!)
```

Add ~30-50% overhead for the index structure itself.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

# Simulated data points for recall vs speed
# (recall, queries_per_second, label, color, marker_size)
algorithms = [
    # Flat variants
    (1.00, 50, 'Flat (exact)', '#e74c3c', 200),
    
    # IVF variants (different nprobe values)
    (0.80, 2000, 'IVF\n(nprobe=1)', '#3498db', 150),
    (0.92, 800, 'IVF\n(nprobe=4)', '#3498db', 150),
    (0.97, 300, 'IVF\n(nprobe=16)', '#3498db', 150),
    (0.99, 120, 'IVF\n(nprobe=64)', '#3498db', 150),
    
    # HNSW variants (different ef values)
    (0.90, 5000, 'HNSW\n(ef=16)', '#2ecc71', 150),
    (0.96, 3000, 'HNSW\n(ef=64)', '#2ecc71', 150),
    (0.99, 1500, 'HNSW\n(ef=128)', '#2ecc71', 150),
    (0.999, 600, 'HNSW\n(ef=512)', '#2ecc71', 150),
]

for recall, qps, label, color, size in algorithms:
    ax.scatter([recall], [qps], c=color, s=size, edgecolors='black',
              linewidth=1.5, zorder=5)
    # Adjust text position for readability
    offset_x, offset_y = 0.005, 0
    ha = 'left'
    if 'Flat' in label:
        offset_x, offset_y = -0.01, -200
        ha = 'right'
    elif 'ef=512' in label:
        offset_x = 0.005
        offset_y = -200
    ax.annotate(label, (recall + offset_x, qps + offset_y),
               fontsize=8, ha=ha, fontweight='bold')

# Connect IVF points with a line
ivf_data = [(r, q) for r, q, l, c, _ in algorithms if 'IVF' in l]
ivf_recalls, ivf_qps = zip(*ivf_data)
ax.plot(ivf_recalls, ivf_qps, '--', color='#3498db', alpha=0.4, linewidth=2)

# Connect HNSW points with a line
hnsw_data = [(r, q) for r, q, l, c, _ in algorithms if 'HNSW' in l]
hnsw_recalls, hnsw_qps = zip(*hnsw_data)
ax.plot(hnsw_recalls, hnsw_qps, '--', color='#2ecc71', alpha=0.4, linewidth=2)

# Ideal zone
from matplotlib.patches import FancyBboxPatch
ideal_box = FancyBboxPatch((0.94, 1200), 0.06, 4000, boxstyle='round,pad=0.01',
                            facecolor='#f39c12', alpha=0.1, edgecolor='#f39c12',
                            linewidth=2, linestyle='--')
ax.add_patch(ideal_box)
ax.text(0.97, 4800, '🎯 Sweet Spot', fontsize=11, ha='center',
        color='#e67e22', fontweight='bold')

ax.set_xlabel('Recall (fraction of true neighbors found)', fontsize=12)
ax.set_ylabel('Queries per Second (QPS)', fontsize=12)
ax.set_title('Recall vs. Speed Tradeoff in Vector Search', fontsize=14, fontweight='bold')
ax.set_xlim(0.75, 1.02)
ax.set_ylim(0, 6000)
ax.grid(True, alpha=0.3)

# Legend
import matplotlib.patches as mpatches
legend_items = [
    mpatches.Patch(color='#e74c3c', label='Flat (Brute Force)'),
    mpatches.Patch(color='#3498db', label='IVF (Cluster-Based)'),
    mpatches.Patch(color='#2ecc71', label='HNSW (Graph-Based)'),
]
ax.legend(handles=legend_items, loc='upper left', fontsize=10)

plt.tight_layout()
plt.savefig('figures/03_recall_vs_speed.png', dpi=150, bbox_inches='tight')
plt.show()
print("Key insight: HNSW gives the best recall-speed tradeoff for most use cases.")
print("IVF is more memory-efficient. Flat is only practical for small datasets.")

---

## ⚠️ Common Misconceptions

### ❌ Misconception 1: "Vector databases replace regular databases"
**Reality:** Vector databases are *complementary* to traditional databases. You still need SQL/NoSQL for structured data, transactions, and exact lookups. Many production systems use both — a vector DB for semantic search and a regular DB for everything else.

### ❌ Misconception 2: "More dimensions = better search quality"
**Reality:** After a certain point, more dimensions lead to the **curse of dimensionality** — all vectors start looking equally far apart, making similarity meaningless. The quality of the embedding *model* matters far more than raw dimension count.

### ❌ Misconception 3: "Approximate search means inaccurate results"
**Reality:** With proper tuning, approximate nearest neighbor (ANN) algorithms achieve **95–99.9% recall** while being 100–1000x faster than exact search. The 0.1% of results you "miss" are usually nearly identical in similarity to the ones you find.

### ❌ Misconception 4: "You need a vector database to do RAG"
**Reality:** For small projects (< 10K documents), you can use a simple numpy array with brute force search and it'll work just fine. Vector databases become essential when you need **scale, persistence, metadata filtering, and production reliability**.

---

## 🎯 Key Takeaways

1. **Vector databases store and search by *meaning***, not exact keywords — they're the backbone of modern semantic search and RAG.

2. **Brute force search is 100% accurate but too slow** for large datasets — we need indexing algorithms.

3. **Flat index** = exact search (small datasets), **IVF** = cluster-based (medium), **HNSW** = graph-based (large, best tradeoff).

4. **Metadata filtering** lets you combine semantic similarity with precise constraints (date, category, etc.).

5. **Recall vs. speed** is the fundamental tradeoff — tune parameters like `nprobe` (IVF) or `ef` (HNSW) to find your sweet spot.

6. **Popular choices:** ChromaDB for prototyping, FAISS for research, Pinecone for managed production, Qdrant/Weaviate for self-hosted.

7. **Memory matters:** 1M vectors at 1536 dimensions needs ~6GB RAM — plan your infrastructure accordingly.

---

## 🧪 Test Your Understanding

<details>
<summary><strong>1. Why can't a regular SQL database find "solar panel efficiency" when you search for "renewable energy"?</strong></summary>

SQL databases use **exact keyword matching**. Since the words "renewable" and "energy" don't appear in "solar panel efficiency", there's no match. A vector database converts both to vectors and finds they're **close in meaning** (high cosine similarity), so it returns the match.
</details>

<details>
<summary><strong>2. If you have 100 million vectors, why is brute force search impractical?</strong></summary>

Brute force requires computing similarity against **every single vector** — that's 100 million distance calculations per query. At 768 dimensions, this takes roughly **10 seconds per query**, which is far too slow for real-time applications.
</details>

<details>
<summary><strong>3. Explain the difference between IVF and HNSW in one sentence each.</strong></summary>

**IVF** divides vectors into clusters and only searches the nearest cluster(s) to the query. **HNSW** builds a multi-layer graph and navigates through connected nodes to quickly reach the query's neighborhood.
</details>

<details>
<summary><strong>4. You have a RAG application with 5,000 documents. Which index type would you use and why?</strong></summary>

**Flat (brute force)** is perfectly fine for 5,000 documents. It gives 100% recall with negligible latency at this scale. The overhead of building an IVF or HNSW index isn't worth it until you reach ~50K+ vectors.
</details>

<details>
<summary><strong>5. What does increasing `nprobe` in IVF do to recall and speed?</strong></summary>

Increasing `nprobe` means searching **more clusters**. This **increases recall** (you find more true nearest neighbors) but **decreases speed** (more vectors to compare). Setting nprobe = total clusters is equivalent to brute force.
</details>

---

## ➡️ What's Next?

Now that you understand how vector databases store and search vectors, the next notebook explores **how to actually retrieve the best documents** for your RAG pipeline:

**[Notebook 04: Retrieval Strategies](04_retrieval_strategies.ipynb)** — Dense retrieval, sparse retrieval, hybrid search, re-ranking, and more!

---

*This notebook is part of the RAG (Retrieval-Augmented Generation) learning series.*